In [1]:
# Loading dataset
from tensorflow.keras.datasets import mnist
(X_train, y_train), (X_test, y_test) = mnist.load_data()

In [2]:
# Device Agnostic code
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.cuda.get_device_name()

'NVIDIA GeForce MX350'

In [3]:
# Dataset class
from torch.utils.data import Dataset, DataLoader

class CustomMNIST(Dataset):
    def __init__(self, features, labels):
        super().__init__()
        self.features = torch.from_numpy(features).to(torch.float32) # NOTE: Datatype of features must be in float format
        self.labels = torch.from_numpy(labels).to(torch.long) # NOTE: Datatype of labels must be in long format (required for CrossEntropyLoss)

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [4]:
# Dataset objects
train_df = CustomMNIST(features=X_train, labels=y_train)
test_df = CustomMNIST(features=X_test, labels=y_test)

In [5]:
# Dataloader for train and test datasets
train_loader = DataLoader(
    dataset=train_df,
    batch_size=32,
    shuffle=True,
    drop_last=False
)

test_loader = DataLoader(
    dataset=test_df,
    batch_size=32,
    shuffle=True,
    drop_last=False
)

In [9]:
# Model Building 
from torch import nn
from torch.nn import Module

class CustomModel(Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=784, out_features=32),
            nn.GELU(),
            nn.Linear(in_features=32, out_features=16),
            nn.GELU(),
            nn.Linear(in_features=16, out_features=10)
        )

        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize weights using He initialization for ReLU networks"""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)

    def forward(self, features):
        return self.model(features)

NOTE → All the functions in this module are intended to be used to initialize neural network parameters, so they all run in torch.no_grad() mode and will not be taken into account by autograd.

1. `torch.nn.init.xavier_uniform_(tensor, gain=1.0, generator=None)` → The resulting tensor will have values sampled from `Uniform(-a, a)`
$$\quad a = \text{gain} \times \sqrt{\frac{6}{\text{fan\_in} + \text{fan\_out}}}$$


2. `torch.nn.init.xavier_normal_(tensor, gain=1.0, generator=None)` → The method is described in Understanding the difficulty of training deep feedforward neural networks - Glorot, X. & Bengio, Y. (2010). The resulting tensor will have values sampled from `Normal(0, std ** 2)`
$$ \text{std} = \text{gain} \times \sqrt{\frac{2}{\text{fan\_in} + \text{fan\_out}}} $$


3. `torch.nn.init.kaiming_uniform_(tensor, a=0, mode='fan_in', nonlinearity='leaky_relu', generator=None)` → The method is described in Delving deep into rectifiers: Surpassing human-level performance on ImageNet classification - He, K. et al. (2015). The resulting tensor will have values sampled from `Uniform(−bound,bound)` <br>
`mode (Literal['fan_in', 'fan_out'])` – either 'fan_in' (default) or 'fan_out'. Choosing 'fan_in' preserves the magnitude of the variance of the weights in the forward pass. Choosing 'fan_out' preserves the magnitudes in the backwards pass.
$$ \text{bound} = \text{gain} \times \sqrt{\frac{3}{\text{fan\_mode}}} $$
​

4. `torch.nn.init.kaiming_normal_(tensor, a=0, mode='fan_in', nonlinearity='leaky_relu', generator=None)` → The method is described in Delving deep into rectifiers: Surpassing human-level performance on ImageNet classification - He, K. et al. (2015). The resulting tensor will have values sampled from `Normal(0,std ** 2)` <br>
`mode (Literal['fan_in', 'fan_out'])` – either 'fan_in' (default) or 'fan_out'. Choosing 'fan_in' preserves the magnitude of the variance of the weights in the forward pass. Choosing 'fan_out' preserves the magnitudes in the backwards pass.
$$
\text{std} = \text{gain} \times \sqrt{\frac{1}{\text{fan\_mode}}}
$$


[PyTorch Documentation - nn.init](https://docs.pytorch.org/docs/2.11/nn.init.html)
 


In [10]:
# Defining parameters
learning_rate = 0.001
epochs = 10

In [11]:
# Model object and shifting to cuda
model = CustomModel().to(device)

In [12]:
# Defining loss functions
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr = learning_rate, betas=(0.9, 0.999), weight_decay = 0.01)

In [13]:
# Training loop
for epoch in range(epochs):
    total_loss = 0
    for batch_features, batch_labels in train_loader:
        # Shifting to GPU
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

        # Forward propogation
        y_pred = model(batch_features)

        # Calculate loss
        loss = criterion(y_pred, batch_labels)

        # Backpropogation
        optimizer.zero_grad()
        loss.backward()

        # Update Gradients
        optimizer.step()

        # Calculating loss
        total_loss += loss.item()
    
    avg_loss = total_loss
    print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')

Epoch: 1 , Loss: 34426.57838201523
Epoch: 2 , Loss: 4331.917881011963
Epoch: 3 , Loss: 4315.688810110092
Epoch: 4 , Loss: 4313.028857707977
Epoch: 5 , Loss: 4318.865425825119
Epoch: 6 , Loss: 3770.6274156570435
Epoch: 7 , Loss: 2610.597877383232
Epoch: 8 , Loss: 831.3491835910827
Epoch: 9 , Loss: 412.6030020453036
Epoch: 10 , Loss: 344.9698043400422


In [14]:
# Enabling evaluation mode
model.eval()

CustomModel(
  (model): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=32, bias=True)
    (2): GELU(approximate='none')
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): GELU(approximate='none')
    (5): Linear(in_features=16, out_features=10, bias=True)
  )
)

`model.eval()` is a **mode switch**. It tells your model: *"Stop learning, I just want to use you now."*

#### What changes after `model.eval()`?

Two specific layer types behave differently depending on whether the model is training or predicting:

##### 1. Dropout Layers
*   **During Training (`model.train()`):** Randomly turns off some neurons to prevent the model from memorizing data. So, two runs of the same input can give slightly different outputs.
*   **After `model.eval()`:** Dropout is **disabled**. Every neuron is active, giving you consistent, reliable predictions.

##### 2. Batch Normalization Layers
*   **During Training:** Uses the statistics (mean, variance) of the **current batch** to normalize data.
*   **After `model.eval()`:** Uses the statistics it **learned and saved during training** instead, making predictions stable and independent of the batch.

---

#### Always pair it with `torch.no_grad()`

`model.eval()` alone does **not** stop gradient calculation. You should always use both together:

| | `model.train()` | `model.eval()` |
|:---|:---:|:---:|
| **Dropout active?** | ✅ Yes | ❌ No |
| **BatchNorm uses batch stats?** | ✅ Yes | ❌ No (uses saved) |
| **Gradients calculated?** | ✅ Yes | ✅ Yes (need `no_grad` to stop) |

In [15]:
# evaluation on test data
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    # Shift data over GPU's
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    # Calculating predictions
    outputs = model(batch_features)

    # Predicted classes
    _, predicted = torch.max(outputs, 1)

    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()

print(correct / total)

0.9445
